# MeMo Torch 
Version integrated with Torch (Keras)

In [2]:
import torch
from MeMoPyTorch.modelling_memo import MeMo
from MeMoPyTorch.modelling_memo_tokenizer import MeMoTokenizer
from MeMoPyTorch.evaluating_memo import Evaluation

Memo: Initializing the Tokenizer and the model

In [3]:
# Meta Parameters : 
#    d - inner dimension
#    h - number of heads
#    l - number of layers
d,h,l = 2048, 4, 3
chunk_length = 4096

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

device = 'cpu'
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    device = 'cuda'

# Intializing Memo 
model = MeMo(inner_dim=d, 
             num_of_heads=h, 
             num_of_layers=l, 
             chunk_length=max_length, 
             num_embeddings=tokenizer.vocab_size, 
             padding_idx=tokenizer.pad_token_id, 
             device=device)


tokenizer_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.


Setting pad token and pad token id = <|endoftext|>, 0
GPU: NVIDIA GeForce GTX 1660 Ti is available.
MeMo embedding initilialization


Reading the two texts

In [4]:
with open("testo_di_prova.txt") as my_first_text_f:
    my_first_text = my_first_text_f.read()
with open("testo_di_prova2.txt") as my_first_text_f:
    my_second_text = my_first_text_f.read()



Memorizing the first text and evaluating if it is memorized

In [5]:
memo_input_1 = tokenizer.get_text_batch_encoding([my_first_text]*1)  # Writing the same doc 8 times to stress the memorization with batch
memo_input_2 = tokenizer.get_text_batch_encoding([my_second_text]*1) # Writing the same doc 8 times to stress the memorization with batch

model.memorize_text(memo_input_1)
e = Evaluation()

e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=8)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=8)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

Starting point : 8


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 4087/4087 [00:13<00:00, 297.64it/s]


Starting point : 8


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 4087/4087 [00:14<00:00, 290.32it/s]

Memorization level of first text  :  tensor(0.9969)
Memorization level of second text :  tensor(0.0357)


Memorizing the second text and checking if it affected the memorization of the first text

In [6]:
model.memorize_text(memo_input_2)

e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=8)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=8)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

Starting point : 8


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 4087/4087 [00:14<00:00, 290.25it/s]


Starting point : 8


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 4087/4087 [00:14<00:00, 290.82it/s]

Memorization level of first text  :  tensor(0.9939)
Memorization level of second text :  tensor(0.9968)


Forgetting the first document

In [7]:
model.forget_text(memo_input_2)

Checking the effect on the two texts

In [8]:
e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=8)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=8)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

Starting point : 8


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 4087/4087 [00:13<00:00, 299.22it/s]


Starting point : 8


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 4087/4087 [00:13<00:00, 297.67it/s]

Memorization level of first text  :  tensor(0.9969)
Memorization level of second text :  tensor(0.0357)
